In [19]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

X_train, y_train = train_df["clean_text"], train_df["sentiment"]
X_test, y_test = test_df["clean_text"], test_df["sentiment"]
labels = ["negative", "neutral", "positive"]
print(len(X_train), len(X_test))

27700 6925


In [20]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
 

dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_test)
 
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.3f} | Macro F1: {f1:.3f}") 
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.933 | Macro F1: 0.322
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00       162
     neutral       0.00      0.00      0.00       300
    positive       0.93      1.00      0.97      6463

    accuracy                           0.93      6925
   macro avg       0.31      0.33      0.32      6925
weighted avg       0.87      0.93      0.90      6925



In [21]:
import re
import numpy as np
from nltk.classify import NaiveBayesClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, classification_report

results = []

# ---------- Block 1: compare models with cross-validation ----------

# 1. Naive Bayes (NLTK)
def to_features(text):
    return {word: True for word in re.findall(r"[a-z']+", text)}

nb_scores = []
for train_idx, val_idx in StratifiedKFold().split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    nb = NaiveBayesClassifier.train([(to_features(t), l) for t, l in zip(X_tr, y_tr)])
    pred = [nb.classify(to_features(t)) for t in X_val]
    nb_scores.append(f1_score(y_val, pred, average="macro"))

results.append({"model": "Naive Bayes (NLTK)", "cv_macro_f1": np.mean(nb_scores)})

# 2-3. SVM and Logistic Regression
models = {
    "SVM": LinearSVC(),
    "Logistic Regression": LogisticRegression(),
}

for name, classifier in models.items():
    pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", classifier)])
    scores = cross_val_score(pipe, X_train, y_train, scoring="f1_macro")
    results.append({"model": name, "cv_macro_f1": scores.mean()})

print(pd.DataFrame(results))

                 model  cv_macro_f1
0   Naive Bayes (NLTK)     0.335751
1                  SVM     0.553970
2  Logistic Regression     0.502585


In [22]:
model = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression())])
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

    negative       0.71      0.22      0.34       162
     neutral       0.45      0.12      0.19       300
    positive       0.95      1.00      0.97      6463

    accuracy                           0.94      6925
   macro avg       0.70      0.44      0.50      6925
weighted avg       0.92      0.94      0.92      6925

